In [39]:
import numpy as np
import pandas as pd
from narwhals import Int64
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, brier_score_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, ParameterGrid, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.base import clone

# 1. Load data

In [38]:
train_df = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_train1.csv", index_col=0)

# 2. Preprocess

In [40]:
# Dropping unnecessary columns for modeling
train_date = train_df["event_dt"]
train_df = train_df.drop(["event_dt", "orig_age", "vital_flag", "orig_height", "orig_weight", "BMI", "year", "Zone", "Latitude", "Longitude", "SubjectID"], axis = 1)

# Unfloating zip code
train_df["Zip"] = train_df["Zip"].astype("Int64")
# Defining categorical columns
cat_cols = ["SubjectRace", "Zip", "District", "StopDescription"]
# Defining discrete/numerical columns
dis_cols = ["SubjectGender", "SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear", "SubjectHasPhotoID", "hour", "night", "isDriver"]
# Numerical columns we would like to scale
scale_cols = ["SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear", "hour"]
# Binary columns
bin_cols = ["SubjectGender", "SubjectHasPhotoID", "night", "isDriver"]

# Converting variables into strings
train_df[cat_cols] = train_df[cat_cols].astype(str)

target = "search_occurred"
X_train = train_df.drop([target], axis = 1)[cat_cols + dis_cols]
y_train = train_df[target]

# 3. Define classes and function for pipeline

In [10]:
class CatBinner(BaseEstimator, TransformerMixin):
    def __init__(self, cols, min_freq=0.01, fill_value="Other"):
        self.cols = cols
        self.min_freq = min_freq
        self.fill_value = fill_value

    def fit(self, X, y=None):
        self.keep_labels = {}
        for c in self.cols:
            freq = X[c].value_counts(normalize=True)
            self.keep_labels[c] = freq[freq >= self.min_freq].index.tolist()

        return self

    def transform(self, X):
        X_copy = X.copy()
        for c in self.cols:
            labels = self.keep_labels[c]
            X_copy[c] = np.where(X_copy[c].isin(labels) | X_copy[c].isna(), X_copy[c], self.fill_value)
        return X_copy

    def set_output(self, *, transform=None):
        return self


class RoundImputed(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        rounded = X[self.cols].round()
        self.min_ = rounded.min()
        self.max_ = rounded.max()
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.cols] = X_copy[self.cols].round().clip(self.min_, self.max_, axis=1)
        return X_copy

    def set_output(self, *, transform=None):
        return self


class OrdEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols, inner):
        self.cat_cols = cat_cols
        self.inner = inner

    def _encode(self, X):
        X_enc = X.copy()
        X_enc[self.cat_cols] = self.encoder_.transform(X[self.cat_cols])
        return X_enc

    def _decode(self, X_imp):
        X_dec = X_imp.copy()
        X_dec[self.cat_cols] = self.encoder_.inverse_transform(X_imp[self.cat_cols])
        return X_dec

    def fit_transform(self, X, y=None):
        self.encoder_ = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)
        self.encoder_.fit(X[self.cat_cols])
        self.inner_ = clone(self.inner)
        return self._decode(self.inner_.fit_transform(self._encode(X), y))

    def fit(self, X, y=None):
        self.fit_transform(X, y)
        return self

    def transform(self, X):
        return self._decode(self.inner_.transform(self._encode(X)))

    def set_output(self, *, transform=None):
        return self


def build_pipeline(model, cat_cols, dis_cols, scale_cols):
    imputer = IterativeImputer(estimator=HistGradientBoostingRegressor(max_iter=50, random_state=10),
                               skip_complete=True, max_iter=5, tol=1e-2, random_state=10)

    impute = Pipeline([
        ("imputer", imputer),
        ("round", RoundImputed(cols=cat_cols + dis_cols)),
    ])

    impute.set_output(transform="pandas")

    encode = ColumnTransformer(
        [("encode", OneHotEncoder(handle_unknown="ignore"), cat_cols), ("scale", StandardScaler(), scale_cols)],
        remainder="passthrough", verbose_feature_names_out=False)

    pipeline = Pipeline([
        ("bin", CatBinner(cols=cat_cols, min_freq=0.01, fill_value="Other")),
        ("imputer", OrdEncoder(cat_cols, impute)),
        ("encode", encode),
        ("model", model)
    ], memory="cache_dir")

    return pipeline

## 3a) Define pipeline

In [41]:
pipeline = build_pipeline(RandomForestClassifier(random_state=10), cat_cols, bin_cols, scale_cols)

# 4. Define random and grid search parameters, run rand search cv
## Because the parameter space has multiple dimensions, it is more computationally efficient to use random search at first to find a range of ideal parameters, and use grid search afterwards on a much tighter space

In [42]:
param_distributions = {"model__criterion":["gini"], "model__max_depth":[None, 3, 6, 10, 20], "model__min_samples_split":randint(2,40), "model__min_samples_leaf":randint(1,40), "model__max_features":["sqrt", "log2", 0.1, 0.3, 0.6]}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

random_search = RandomizedSearchCV(cv = cv, estimator=pipeline, param_distributions= param_distributions, scoring=["roc_auc", "neg_brier_score", "average_precision"], refit = "average_precision", verbose=2, random_state=10, n_iter=10)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END model__criterion=gini, model__max_depth=3, model__max_features=0.6, model__min_samples_leaf=16, model__min_samples_split=2; total time=   5.5s
[CV] END model__criterion=gini, model__max_depth=3, model__max_features=0.6, model__min_samples_leaf=16, model__min_samples_split=2; total time=   4.9s
[CV] END model__criterion=gini, model__max_depth=3, model__max_features=0.6, model__min_samples_leaf=16, model__min_samples_split=2; total time=   9.3s
[CV] END model__criterion=gini, model__max_depth=3, model__max_features=0.6, model__min_samples_leaf=16, model__min_samples_split=2; total time=   6.9s
[CV] END model__criterion=gini, model__max_depth=3, model__max_features=0.6, model__min_samples_leaf=16, model__min_samples_split=2; total time=   7.7s
[CV] END model__criterion=gini, model__max_depth=3, model__max_features=0.3, model__min_samples_leaf=29, model__min_samples_split=27; total time=   6.6s
[CV] END model__criterion=

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(memo...m_state=10))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__criterion': ['gini'], 'model__max_depth': [None, 3, ...], 'model__max_features': ['sqrt', 'log2', ...], 'model__min_samples_leaf': <scipy.stats....0027110BB1130>, ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","['roc_auc', 'neg_brier_score', ...]"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'average_precision'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting

In [43]:
# Save results
rand_results = pd.DataFrame(random_search.cv_results_)
rand_results.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\rf_hyp_tuning.csv")

In [45]:
# Best parameters based on:
## Balanced accuracy
## Precision
print("PRAUC/Average Precision:", rand_results["params"][rand_results["rank_test_average_precision"] == 1].head(1).item())
print("AUC:", rand_results["params"][rand_results["rank_test_roc_auc"] == 1].head(1).item())
## Brier
print("Brier score:", rand_results["params"][rand_results["rank_test_neg_brier_score"] == 1].head(1).item())

PRAUC/Average Precision: {'model__criterion': 'gini', 'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 9, 'model__min_samples_split': 11}
AUC: {'model__criterion': 'gini', 'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 9, 'model__min_samples_split': 11}
Brier score: {'model__criterion': 'gini', 'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 9, 'model__min_samples_split': 11}


## 4a) Grid search on smaller parameter space

In [46]:
# Let's try no max depth
param_grid = {"model__criterion":["gini"], "model__max_depth":[None], "model__min_samples_split":[11, 12, 13], "model__min_samples_leaf":[9, 10, 11], "model__max_features":["log2"]}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

grid_search = GridSearchCV(cv = cv, estimator=pipeline, param_grid= param_grid, scoring=["roc_auc", "neg_brier_score", "average_precision"], refit = "average_precision", verbose=2)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV] END model__criterion=gini, model__max_depth=None, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=11; total time=  56.6s
[CV] END model__criterion=gini, model__max_depth=None, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=11; total time= 1.0min
[CV] END model__criterion=gini, model__max_depth=None, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=11; total time= 1.4min
[CV] END model__criterion=gini, model__max_depth=None, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=11; total time= 1.0min
[CV] END model__criterion=gini, model__max_depth=None, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=11; total time= 1.1min
[CV] END model__criterion=gini, model__max_depth=None, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=12; total time= 1.0min
[CV]

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(memo...m_state=10))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__criterion': ['gini'], 'model__max_depth': [None], 'model__max_features': ['log2'], 'model__min_samples_leaf': [9, 10, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","['roc_auc', 'neg_brier_score', ...]"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'average_precision'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total n

In [47]:
# Save grid search results
grid_results = pd.DataFrame(grid_search.cv_results_)
grid_results.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\rf_hyp_tuning_grid.csv")

In [48]:
# Best parameters based on:
## Balanced accuracy
## Precision
print("PRAUC/Average Precision:", grid_results["params"][grid_results["rank_test_average_precision"] == 1].head(1).item())
print("AUC:", grid_results["params"][grid_results["rank_test_roc_auc"] == 1].head(1).item())
## Brier
print("Brier score:", grid_results["params"][grid_results["rank_test_neg_brier_score"] == 1].head(1).item())

PRAUC/Average Precision: {'model__criterion': 'gini', 'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 9, 'model__min_samples_split': 11}
AUC: {'model__criterion': 'gini', 'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 10, 'model__min_samples_split': 11}
Brier score: {'model__criterion': 'gini', 'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 9, 'model__min_samples_split': 11}


In [49]:
# Implement one-standard-error rule from Elements of Statistical Learning
# Find index of obs with highest mean test average PR-AUC
ind_opt = grid_results["mean_test_average_precision"].idxmax()
# Find according mean and sd
mean_opt = grid_results["mean_test_average_precision"].loc[ind_opt]
se_opt = grid_results["std_test_average_precision"].loc[ind_opt]
# Find rows where the performance is within the acceptable range
opt_grid_results = grid_results[(mean_opt - se_opt <= grid_results["mean_test_average_precision"]) & (grid_results["mean_test_average_precision"] <= mean_opt + se_opt)]
# Ideal parameters are the largest min_samples_leaf and largest min_samples_split
# Pick the row with the largest values above
opt_depth = opt_grid_results["param_model__max_depth"][(opt_grid_results["param_model__min_samples_leaf"] == max(opt_grid_results["param_model__min_samples_leaf"])) & (opt_grid_results["param_model__min_samples_split"] == max(opt_grid_results["param_model__min_samples_split"]))].item()

opt_min_samples_leaf = opt_grid_results["param_model__min_samples_leaf"][(opt_grid_results["param_model__min_samples_leaf"] == max(opt_grid_results["param_model__min_samples_leaf"])) & (opt_grid_results["param_model__min_samples_split"] == max(opt_grid_results["param_model__min_samples_split"]))].item()

opt_min_samples_split = opt_grid_results["param_model__min_samples_split"][(opt_grid_results["param_model__min_samples_leaf"] == max(opt_grid_results["param_model__min_samples_leaf"])) & (opt_grid_results["param_model__min_samples_split"] == max(opt_grid_results["param_model__min_samples_split"]))].item()

opt_max_features = opt_grid_results["param_model__max_features"][(opt_grid_results["param_model__min_samples_leaf"] == max(opt_grid_results["param_model__min_samples_leaf"])) & (opt_grid_results["param_model__min_samples_split"] == max(opt_grid_results["param_model__min_samples_split"]))].item()

print(f"Optimal parameters: criterion = gini, max_depth:{opt_depth}, min_samples_split:{opt_min_samples_split}, min_samples_leaf:{opt_min_samples_leaf}, max_features:{opt_max_features}")

Optimal parameters: criterion = gini, max_depth:None, min_samples_split:13, min_samples_leaf:11, max_features:log2


## Using random search and grid search and the PR-AUC metric and the one-standard-error rule, the optimal parameters are criterion = "gini", max_depth = None, min_samples_split = 13, min_samples_leaf = 11, max_features = "log2"